# Phase 3+4 - YOLO26-seg

Owner: Nguyen Ho Anh Tuan. This Kaggle notebook converts the project COCO annotations into Ultralytics YOLO segmentation format, trains YOLO26-seg, evaluates/tunes it, exports the selected checkpoint, and optionally uploads artifacts to HuggingFace Hub.

Official references:
- YOLO26: https://docs.ultralytics.com/models/yolo26
- Instance segmentation: https://docs.ultralytics.com/tasks/segment
- HuggingFace upload: https://huggingface.co/docs/huggingface_hub/en/guides/upload


## 0. Install optional Kaggle dependencies

Local runs should use the uv-locked environment (`uv sync --frozen`). Kaggle runs can use this cell to install missing runtime packages in the active notebook kernel.

In [ ]:
import importlib.util
import subprocess
import sys

required_packages = {
    "ultralytics": "ultralytics",
    "huggingface_hub": "huggingface_hub",
    "pycocotools": "pycocotools",
}
missing = [pkg for module, pkg in required_packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("Training dependencies already available.")


## 1. Configuration and taxonomy

The PLAN taxonomy is the source of truth. Coffee labels are derived from raw folder ids (`0..3`) so raw COCO annotations and processed manifests stay aligned; `label_idx` is treated as auxiliary metadata.

In [ ]:
from __future__ import annotations

import json
import os
import random
import shutil
import time
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import yaml
from PIL import Image, ImageDraw

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

def find_project_root() -> Path:
    env_root = os.environ.get("PROJECT_ROOT")
    candidates = []
    if env_root:
        candidates.append(Path(env_root))
    candidates.append(Path("/kaggle/input/ml-vietnam-plant-disease-detection"))
    cwd = Path.cwd().resolve()
    candidates.extend([cwd, *cwd.parents])
    for candidate in candidates:
        if (candidate / "data" / "raw").exists() and (candidate / "models" / "class_names.json").exists():
            return candidate
    raise FileNotFoundError("Could not locate project root. Set PROJECT_ROOT explicitly.")


PROJECT_ROOT = find_project_root()

RAW_ROOT = Path(os.environ.get("RAW_DATA_ROOT", PROJECT_ROOT / "data" / "raw"))
MANIFEST_DIR = Path(os.environ.get("MANIFEST_DIR", PROJECT_ROOT / "data" / "processed" / "metadata"))
CLASS_NAMES_PATH = Path(os.environ.get("CLASS_NAMES_PATH", PROJECT_ROOT / "models" / "class_names.json"))

DEFAULT_WORK_DIR = Path("/kaggle/working/yolo26_seg_phase34") if Path("/kaggle/working").exists() else PROJECT_ROOT / "notebooks" / "models" / "yolo26_seg" / "_working"
WORK_DIR = Path(os.environ.get("WORK_DIR", DEFAULT_WORK_DIR))
DATASET_DIR = WORK_DIR / "dataset"
RUNS_DIR = WORK_DIR / "runs"
ARTIFACTS_DIR = WORK_DIR / "artifacts"
for directory in [DATASET_DIR, RUNS_DIR, ARTIFACTS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)
YOLO_CONFIG_ROOT = WORK_DIR / "config"
YOLO_CONFIG_ROOT.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("YOLO_CONFIG_DIR", str(YOLO_CONFIG_ROOT))

PLAN_CLASS_NAMES = [
    "Healthy",
    "BrownSpot",
    "Hispa",
    "LeafBlast",
    "LeafMiner",
    "PowderyMildew",
    "Rust",
    "AlgalLeafSpot",
]
if CLASS_NAMES_PATH.exists():
    CLASS_NAMES = json.loads(CLASS_NAMES_PATH.read_text(encoding="utf-8"))
else:
    CLASS_NAMES = PLAN_CLASS_NAMES
assert CLASS_NAMES == PLAN_CLASS_NAMES, f"Class order must match PLAN: {CLASS_NAMES}"
CLASS_TO_ID = {name: idx for idx, name in enumerate(CLASS_NAMES)}

RICE_CLASSES = {"Healthy", "BrownSpot", "Hispa", "LeafBlast"}
COFFEE_FOLDER_TO_CLASS = {
    "0": "LeafMiner",
    "1": "PowderyMildew",
    "2": "Rust",
    "3": "AlgalLeafSpot",
}
CONFUSION_PAIRS = [("BrownSpot", "LeafBlast"), ("AlgalLeafSpot", "Rust")]

DEVICE = 0 if torch.cuda.is_available() else "cpu"
print(f"PROJECT_ROOT={PROJECT_ROOT}")
print(f"RAW_ROOT={RAW_ROOT}")
print(f"MANIFEST_DIR={MANIFEST_DIR}")
print(f"DEVICE={DEVICE}")
print(CLASS_TO_ID)


## 2. Load manifests and COCO annotations

The split manifests remain fixed to avoid train/validation/test leakage. COCO raw annotations are used because their polygon coordinates match raw images.

In [ ]:
def read_manifest(split: str) -> pd.DataFrame:
    path = MANIFEST_DIR / f"{split}_manifest.csv"
    if not path.exists():
        raise FileNotFoundError(f"Missing manifest: {path}")
    df = pd.read_csv(path)
    df["split"] = split
    return df


manifest_df = pd.concat([read_manifest(split) for split in ["train", "val", "test"]], ignore_index=True)
print(manifest_df.groupby(["split", "domain", "label"]).size())


def assert_no_manifest_leakage(df: pd.DataFrame) -> None:
    for field in ["sample_id", "md5"]:
        split_sets = {split: set(part[field].astype(str)) for split, part in df.groupby("split")}
        for left, right in [("train", "val"), ("train", "test"), ("val", "test")]:
            overlap = split_sets[left] & split_sets[right]
            assert not overlap, f"Leakage in {field}: {left}-{right} has {len(overlap)} overlaps"
    print("No manifest leakage by sample_id or md5.")


assert_no_manifest_leakage(manifest_df)


def load_json(path: Path) -> dict:
    if not path.exists():
        raise FileNotFoundError(f"Missing COCO annotation file: {path}")
    with path.open("r", encoding="utf-8") as handle:
        return json.load(handle)


rice_coco = load_json(RAW_ROOT / "rice_leaf_disease" / "annotations.coco.json")
coffee_coco = load_json(RAW_ROOT / "coffee_leaf_disease" / "annotations.coco.json")
print(f"Rice COCO images={len(rice_coco['images'])}, annotations={len(rice_coco['annotations'])}")
print(f"Coffee COCO images={len(coffee_coco['images'])}, annotations={len(coffee_coco['annotations'])}")


## 3. Convert COCO polygons to YOLO-seg

Images without valid polygon masks are excluded. Each YOLO label row is `class_id x1 y1 x2 y2 ...` with normalized coordinates in `[0, 1]`.

In [ ]:
@dataclass
class CocoIndex:
    images_by_file: dict[str, dict]
    annotations_by_image: dict[int, list[dict]]
    category_to_class: dict[int, str]


def norm_path(value: str) -> str:
    return str(value).replace("\\", "/").strip().lower()


def has_polygon(segmentation) -> bool:
    return isinstance(segmentation, list) and any(isinstance(poly, list) and len(poly) >= 6 for poly in segmentation)


def build_coco_index(coco: dict, domain: str) -> CocoIndex:
    if domain == "rice":
        category_to_class = {
            int(cat["id"]): cat["name"]
            for cat in coco.get("categories", [])
            if cat.get("name") in RICE_CLASSES
        }
    elif domain == "coffee":
        category_to_class = {
            int(cat["id"]): COFFEE_FOLDER_TO_CLASS[str(cat["name"])]
            for cat in coco.get("categories", [])
            if str(cat.get("name")) in COFFEE_FOLDER_TO_CLASS
        }
    else:
        raise ValueError(domain)

    images_by_file = {norm_path(img["file_name"]): img for img in coco.get("images", [])}
    annotations_by_image: dict[int, list[dict]] = defaultdict(list)
    for ann in coco.get("annotations", []):
        category_id = int(ann.get("category_id", -1))
        if category_id not in category_to_class:
            continue
        if not has_polygon(ann.get("segmentation")):
            continue
        annotations_by_image[int(ann["image_id"])].append(ann)
    return CocoIndex(images_by_file, dict(annotations_by_image), category_to_class)


COCO_BY_DOMAIN = {
    "rice": build_coco_index(rice_coco, "rice"),
    "coffee": build_coco_index(coffee_coco, "coffee"),
}


def coco_file_from_row(row: pd.Series) -> str:
    rel = norm_path(row["relative_path"])
    marker = "rice_leaf_disease/" if row["domain"] == "rice" else "coffee_leaf_disease/"
    if marker not in rel:
        return f"{row['class_folder']}/{row['file_name']}"
    return rel.split(marker, 1)[1]


def raw_image_path(domain: str, coco_file_name: str) -> Path:
    raw_folder = "rice_leaf_disease" if domain == "rice" else "coffee_leaf_disease"
    return RAW_ROOT / raw_folder / coco_file_name


def polygon_to_yolo(poly: list[float], width: int, height: int) -> list[float] | None:
    if len(poly) < 6 or len(poly) % 2 != 0:
        return None
    coords = np.asarray(poly, dtype=np.float32).reshape(-1, 2)
    if coords.shape[0] < 3:
        return None
    coords[:, 0] = np.clip(coords[:, 0] / float(width), 0.0, 1.0)
    coords[:, 1] = np.clip(coords[:, 1] / float(height), 0.0, 1.0)
    if np.unique(coords, axis=0).shape[0] < 3:
        return None
    return coords.reshape(-1).tolist()


def reset_yolo_dataset(root: Path) -> None:
    if root.exists():
        shutil.rmtree(root)
    for split in ["train", "val", "test"]:
        (root / "images" / split).mkdir(parents=True, exist_ok=True)
        (root / "labels" / split).mkdir(parents=True, exist_ok=True)


def convert_to_yolo_seg(manifest: pd.DataFrame, dataset_root: Path) -> pd.DataFrame:
    reset_yolo_dataset(dataset_root)
    rows: list[dict] = []
    stats = defaultdict(int)

    for _, row in manifest.iterrows():
        domain = row["domain"]
        split = row["split"]
        index = COCO_BY_DOMAIN[domain]
        coco_file = coco_file_from_row(row)
        image_entry = index.images_by_file.get(norm_path(coco_file))
        if image_entry is None:
            stats[f"{split}_missing_coco_image"] += 1
            continue

        annotations = index.annotations_by_image.get(int(image_entry["id"]), [])
        if not annotations:
            stats[f"{split}_no_polygon_mask"] += 1
            continue

        source = raw_image_path(domain, coco_file)
        if not source.exists():
            stats[f"{split}_missing_image_file"] += 1
            continue

        width = int(image_entry["width"])
        height = int(image_entry["height"])
        label_lines: list[str] = []
        class_names_for_image: set[str] = set()
        for ann in annotations:
            class_name = index.category_to_class[int(ann["category_id"])]
            class_id = CLASS_TO_ID[class_name]
            for poly in ann.get("segmentation", []):
                yolo_poly = polygon_to_yolo(poly, width=width, height=height)
                if yolo_poly is None:
                    stats[f"{split}_invalid_polygon"] += 1
                    continue
                values = [str(class_id), *[f"{v:.6f}" for v in yolo_poly]]
                label_lines.append(" ".join(values))
                class_names_for_image.add(class_name)

        if not label_lines:
            stats[f"{split}_no_valid_segments"] += 1
            continue

        suffix = source.suffix.lower() or str(row.get("file_ext", ".jpg"))
        target_stem = str(row["sample_id"])
        target_image = dataset_root / "images" / split / f"{target_stem}{suffix}"
        target_label = dataset_root / "labels" / split / f"{target_stem}.txt"
        shutil.copy2(source, target_image)
        target_label.write_text("\n".join(label_lines) + "\n", encoding="utf-8")

        rows.append({
            "sample_id": row["sample_id"],
            "split": split,
            "domain": domain,
            "source_coco_file": coco_file,
            "image_path": str(target_image),
            "label_path": str(target_label),
            "width": width,
            "height": height,
            "num_segments": len(label_lines),
            "classes": ";".join(sorted(class_names_for_image)),
            "md5": row["md5"],
        })
        stats[f"{split}_converted"] += 1

    stats_df = pd.Series(dict(stats), name="count").sort_index().to_frame()
    display(stats_df)
    converted = pd.DataFrame(rows)
    if converted.empty:
        raise RuntimeError("No YOLO-seg samples were converted. Check raw data paths and COCO files.")
    return converted


converted_df = convert_to_yolo_seg(manifest_df, DATASET_DIR)
display(converted_df.groupby(["split", "domain", "classes"]).size().rename("n").reset_index().head(20))
converted_df.to_csv(WORK_DIR / "converted_yolo_seg_manifest.csv", index=False)

data_yaml = {
    "path": str(DATASET_DIR),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": {idx: name for idx, name in enumerate(CLASS_NAMES)},
}
DATA_YAML = DATASET_DIR / "data.yaml"
DATA_YAML.write_text(yaml.safe_dump(data_yaml, sort_keys=False, allow_unicode=True), encoding="utf-8")
print(DATA_YAML.read_text(encoding="utf-8"))


## 4. Conversion validation and ground-truth overlays

In [ ]:
def read_yolo_label(label_path: Path) -> list[tuple[int, np.ndarray]]:
    segments: list[tuple[int, np.ndarray]] = []
    for line in label_path.read_text(encoding="utf-8").splitlines():
        parts = line.strip().split()
        if not parts:
            continue
        class_id = int(parts[0])
        coords = np.asarray([float(v) for v in parts[1:]], dtype=np.float32).reshape(-1, 2)
        segments.append((class_id, coords))
    return segments


def validate_yolo_labels(df: pd.DataFrame) -> None:
    for _, row in df.iterrows():
        label_path = Path(row["label_path"])
        segments = read_yolo_label(label_path)
        assert segments, f"Empty label file: {label_path}"
        for class_id, coords in segments:
            assert 0 <= class_id < len(CLASS_NAMES), f"Invalid class_id {class_id} in {label_path}"
            assert coords.shape[0] >= 3, f"Polygon has <3 points in {label_path}"
            assert np.all(coords >= 0.0) and np.all(coords <= 1.0), f"Out-of-range polygon in {label_path}"

    split_sets = {split: set(part["md5"].astype(str)) for split, part in df.groupby("split")}
    for left, right in [("train", "val"), ("train", "test"), ("val", "test")]:
        overlap = split_sets[left] & split_sets[right]
        assert not overlap, f"Converted leakage by md5: {left}-{right} has {len(overlap)} overlaps"
    print(f"Validated {len(df)} converted images and all YOLO polygons.")


validate_yolo_labels(converted_df)
display(converted_df.groupby("split")["num_segments"].agg(["count", "sum", "mean"]))


PALETTE = [
    "#1B9E77", "#D95F02", "#7570B3", "#E7298A",
    "#66A61E", "#E6AB02", "#A6761D", "#666666",
]


def overlay_segments(image_path: Path, label_path: Path, alpha: int = 95) -> Image.Image:
    image = Image.open(image_path).convert("RGBA")
    width, height = image.size
    overlay = Image.new("RGBA", image.size, (0, 0, 0, 0))
    draw = ImageDraw.Draw(overlay)
    for class_id, coords in read_yolo_label(label_path):
        points = [(float(x) * width, float(y) * height) for x, y in coords]
        rgb = tuple(int(PALETTE[class_id].lstrip("#")[i:i + 2], 16) for i in (0, 2, 4))
        draw.polygon(points, fill=rgb + (alpha,), outline=rgb + (255,))
        draw.text(points[0], CLASS_NAMES[class_id], fill=rgb + (255,))
    return Image.alpha_composite(image, overlay).convert("RGB")


def plot_ground_truth_samples(df: pd.DataFrame, n: int = 20, split: str = "train") -> None:
    sample = df[df["split"] == split].sample(min(n, (df["split"] == split).sum()), random_state=SEED)
    cols = 4
    rows = int(np.ceil(len(sample) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 4))
    axes = np.asarray(axes).reshape(-1)
    for ax, (_, item) in zip(axes, sample.iterrows()):
        ax.imshow(overlay_segments(Path(item["image_path"]), Path(item["label_path"])))
        ax.set_title(f"{item['split']} | {item['classes']}", fontsize=9)
        ax.axis("off")
    for ax in axes[len(sample):]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()


plot_ground_truth_samples(converted_df, n=20, split="train")


## 5. Train: smoke test, baseline, tuned config

The smoke test should run first to validate the dataset YAML and dataloader. Full training includes the required baseline and tuned configs. Set `RUN_OPTIONAL_YOLO26S=1` to add the optional YOLO26s-seg run when GPU budget allows.

In [ ]:
from ultralytics import YOLO

RUN_SMOKE_TEST = os.environ.get("RUN_SMOKE_TEST", "1") == "1"
RUN_FULL_TRAINING = os.environ.get("RUN_FULL_TRAINING", "1") == "1"
RUN_OPTIONAL_YOLO26S = os.environ.get("RUN_OPTIONAL_YOLO26S", "0") == "1"

COMMON_ARGS = {
    "data": str(DATA_YAML),
    "imgsz": 640,
    "batch": int(os.environ.get("YOLO_BATCH", "8")),
    "workers": int(os.environ.get("YOLO_WORKERS", "2")),
    "seed": SEED,
    "deterministic": True,
    "project": str(RUNS_DIR),
    "device": DEVICE,
    "exist_ok": True,
    "plots": True,
    "save": True,
    "val": True,
}

EXPERIMENTS = [
    {
        "name": "baseline_yolo26n_seg",
        "model": "yolo26n-seg.pt",
        "epochs": 50,
        "optimizer": "auto",
        "patience": 15,
        "degrees": 15.0,
        "translate": 0.08,
        "scale": 0.20,
        "fliplr": 0.50,
        "flipud": 0.10,
        "hsv_h": 0.015,
        "hsv_s": 0.50,
        "hsv_v": 0.30,
    },
    {
        "name": "tuned_yolo26n_seg",
        "model": "yolo26n-seg.pt",
        "epochs": 80,
        "optimizer": "auto",
        "patience": 25,
        "lr0": 0.003,
        "lrf": 0.01,
        "degrees": 25.0,
        "translate": 0.10,
        "scale": 0.30,
        "fliplr": 0.50,
        "flipud": 0.20,
        "hsv_h": 0.020,
        "hsv_s": 0.60,
        "hsv_v": 0.40,
        "copy_paste": 0.10,
    },
]
if RUN_OPTIONAL_YOLO26S:
    EXPERIMENTS.append({**EXPERIMENTS[-1], "name": "optional_tuned_yolo26s_seg", "model": "yolo26s-seg.pt"})


def extract_ultralytics_metrics(metrics) -> dict:
    def value(path: str):
        current = metrics
        for part in path.split("."):
            current = getattr(current, part, None)
            if current is None:
                return None
        try:
            return float(current)
        except TypeError:
            return current

    return {
        "mAP50_mask": value("seg.map50"),
        "mAP50_95_mask": value("seg.map"),
        "mAP75_mask": value("seg.map75"),
        "mAP50_box": value("box.map50"),
        "mAP50_95_box": value("box.map"),
    }


def train_one_experiment(config: dict) -> dict:
    model = YOLO(config["model"])
    train_args = {**COMMON_ARGS, **{k: v for k, v in config.items() if k not in {"model"}}}
    print(f"Training {config['name']} with {config['model']}")
    train_result = model.train(**train_args)
    save_dir = Path(getattr(train_result, "save_dir", model.trainer.save_dir))
    best_ckpt = save_dir / "weights" / "best.pt"
    if not best_ckpt.exists():
        raise FileNotFoundError(best_ckpt)

    val_model = YOLO(str(best_ckpt))
    test_metrics = val_model.val(
        data=str(DATA_YAML),
        split="test",
        imgsz=train_args["imgsz"],
        batch=1,
        device=DEVICE,
        project=str(RUNS_DIR),
        name=f"{config['name']}_test",
        plots=True,
        exist_ok=True,
    )
    summary = {
        "run": config["name"],
        "model": config["model"],
        "epochs": config["epochs"],
        "imgsz": train_args["imgsz"],
        "best_ckpt": str(best_ckpt),
        "save_dir": str(save_dir),
        "size_mb": best_ckpt.stat().st_size / (1024 * 1024),
        **extract_ultralytics_metrics(test_metrics),
    }
    return summary


if RUN_SMOKE_TEST:
    smoke = YOLO("yolo26n-seg.pt")
    smoke.train(
        **COMMON_ARGS,
        name="smoke_yolo26n_seg",
        epochs=3,
        batch=max(1, min(4, COMMON_ARGS["batch"])),
        patience=2,
        fraction=float(os.environ.get("SMOKE_FRACTION", "0.1")),
    )

experiment_results: list[dict] = []
if RUN_FULL_TRAINING:
    for experiment in EXPERIMENTS:
        experiment_results.append(train_one_experiment(experiment))

    metric_table = pd.DataFrame(experiment_results)
    display(metric_table)
    metric_table.to_csv(ARTIFACTS_DIR / "yolo26_seg_experiment_metrics.csv", index=False)
else:
    metric_table = pd.DataFrame()
    print("RUN_FULL_TRAINING=0, skipping full training.")


## 6. Custom mask mIoU, Dice, speed benchmark, and visualizations

Ultralytics reports segmentation mAP. The project also requires mIoU, Dice, inference time, and test overlays.

In [ ]:
def rasterize_label_file(label_path: Path, width: int, height: int) -> np.ndarray:
    masks = np.zeros((len(CLASS_NAMES), height, width), dtype=bool)
    for class_id, coords in read_yolo_label(label_path):
        pixel_points = [(float(x) * width, float(y) * height) for x, y in coords]
        mask_img = Image.new("L", (width, height), 0)
        ImageDraw.Draw(mask_img).polygon(pixel_points, outline=1, fill=1)
        masks[class_id] |= np.asarray(mask_img, dtype=bool)
    return masks


def rasterize_predictions(result, width: int, height: int) -> np.ndarray:
    masks = np.zeros((len(CLASS_NAMES), height, width), dtype=bool)
    if result.masks is None or result.boxes is None or len(result.boxes) == 0:
        return masks
    pred_masks = result.masks.data.detach().cpu().numpy()
    pred_classes = result.boxes.cls.detach().cpu().numpy().astype(int)
    for class_id, mask in zip(pred_classes, pred_masks):
        if not 0 <= int(class_id) < len(CLASS_NAMES):
            continue
        mask_bool = mask > 0.5
        if mask_bool.shape != (height, width):
            mask_img = Image.fromarray((mask_bool.astype(np.uint8) * 255)).resize((width, height), Image.Resampling.NEAREST)
            mask_bool = np.asarray(mask_img, dtype=np.uint8) > 0
        masks[int(class_id)] |= mask_bool
    return masks


def mean_iou_dice(gt: np.ndarray, pred: np.ndarray) -> tuple[float, float, dict, list[str], list[str]]:
    ious = []
    dices = []
    per_class = {}
    gt_classes = []
    pred_classes = []
    for class_id, class_name in enumerate(CLASS_NAMES):
        gt_mask = gt[class_id]
        pred_mask = pred[class_id]
        if gt_mask.any():
            gt_classes.append(class_name)
        if pred_mask.any():
            pred_classes.append(class_name)
        if not gt_mask.any() and not pred_mask.any():
            continue
        intersection = np.logical_and(gt_mask, pred_mask).sum()
        union = np.logical_or(gt_mask, pred_mask).sum()
        denom = gt_mask.sum() + pred_mask.sum()
        iou = float(intersection / union) if union else 1.0
        dice = float((2 * intersection) / denom) if denom else 1.0
        ious.append(iou)
        dices.append(dice)
        per_class[class_name] = {"iou": iou, "dice": dice}
    return float(np.mean(ious) if ious else 0.0), float(np.mean(dices) if dices else 0.0), per_class, gt_classes, pred_classes


def evaluate_custom_masks(model_path: Path, test_df: pd.DataFrame, imgsz: int = 640, max_images: int | None = None) -> tuple[dict, pd.DataFrame]:
    model = YOLO(str(model_path))
    items = test_df[test_df["split"] == "test"].copy().reset_index(drop=True)
    if max_images is not None:
        items = items.head(max_images)
    rows = []
    for _, item in items.iterrows():
        image_path = Path(item["image_path"])
        label_path = Path(item["label_path"])
        with Image.open(image_path) as img:
            width, height = img.size
        gt_masks = rasterize_label_file(label_path, width, height)
        result = model.predict(str(image_path), imgsz=imgsz, device=DEVICE, conf=0.25, retina_masks=True, verbose=False)[0]
        pred_masks = rasterize_predictions(result, width, height)
        image_iou, image_dice, per_class, gt_classes, pred_classes = mean_iou_dice(gt_masks, pred_masks)
        pair_error = any((a in gt_classes and b in pred_classes) or (b in gt_classes and a in pred_classes) for a, b in CONFUSION_PAIRS)
        rows.append({
            "sample_id": item["sample_id"],
            "image_path": str(image_path),
            "label_path": str(label_path),
            "mIoU": image_iou,
            "Dice": image_dice,
            "gt_classes": ";".join(gt_classes),
            "pred_classes": ";".join(pred_classes),
            "pair_error": pair_error,
            "per_class": per_class,
        })
    score_df = pd.DataFrame(rows)
    summary = {"mIoU": float(score_df["mIoU"].mean()), "Dice": float(score_df["Dice"].mean()), "custom_eval_images": int(len(score_df))}
    return summary, score_df


def benchmark_inference(model_path: Path, test_df: pd.DataFrame, imgsz: int = 640, n_images: int = 50) -> float:
    model = YOLO(str(model_path))
    paths = [str(p) for p in test_df[test_df["split"] == "test"]["image_path"].head(n_images)]
    if not paths:
        return float("nan")
    for path in paths[:5]:
        model.predict(path, imgsz=imgsz, device=DEVICE, conf=0.25, verbose=False)
    start = time.perf_counter()
    for path in paths:
        model.predict(path, imgsz=imgsz, device=DEVICE, conf=0.25, verbose=False)
    elapsed = time.perf_counter() - start
    return float(1000 * elapsed / len(paths))


def draw_mask_array(draw: ImageDraw.ImageDraw, mask: np.ndarray, color: tuple[int, int, int, int]) -> None:
    mask_img = Image.fromarray((mask.astype(np.uint8) * color[3]), mode="L")
    color_img = Image.new("RGBA", mask_img.size, color)
    draw.bitmap((0, 0), mask_img, fill=color)


def overlay_prediction(image_path: Path, label_path: Path, result) -> Image.Image:
    image = Image.open(image_path).convert("RGBA")
    width, height = image.size
    overlay = Image.new("RGBA", image.size, (0, 0, 0, 0))
    draw = ImageDraw.Draw(overlay)
    # Ground truth: green polygons.
    for class_id, coords in read_yolo_label(label_path):
        points = [(float(x) * width, float(y) * height) for x, y in coords]
        draw.polygon(points, fill=(30, 180, 90, 75), outline=(30, 180, 90, 255))
        draw.text(points[0], f"GT {CLASS_NAMES[class_id]}", fill=(20, 120, 60, 255))
    # Predictions: red mask contours/fill.
    pred_masks = rasterize_predictions(result, width, height)
    for class_id, mask in enumerate(pred_masks):
        if mask.any():
            mask_img = Image.fromarray((mask.astype(np.uint8) * 95), mode="L")
            red = Image.new("RGBA", image.size, (220, 40, 40, 95))
            overlay.alpha_composite(Image.composite(red, Image.new("RGBA", image.size, (0, 0, 0, 0)), mask_img))
            ys, xs = np.where(mask)
            draw.text((float(xs.min()), float(ys.min())), f"P {CLASS_NAMES[class_id]}", fill=(180, 20, 20, 255))
    return Image.alpha_composite(image, overlay).convert("RGB")


def plot_prediction_samples(model_path: Path, test_df: pd.DataFrame, n: int = 10, title: str = "Predictions vs GT") -> None:
    model = YOLO(str(model_path))
    sample = test_df[test_df["split"] == "test"].sample(min(n, (test_df["split"] == "test").sum()), random_state=SEED)
    cols = 2
    rows = int(np.ceil(len(sample) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 6, rows * 5))
    axes = np.asarray(axes).reshape(-1)
    for ax, (_, item) in zip(axes, sample.iterrows()):
        result = model.predict(item["image_path"], imgsz=640, device=DEVICE, conf=0.25, retina_masks=True, verbose=False)[0]
        ax.imshow(overlay_prediction(Path(item["image_path"]), Path(item["label_path"]), result))
        ax.set_title(f"{item['sample_id']} | {item['classes']}", fontsize=9)
        ax.axis("off")
    for ax in axes[len(sample):]:
        ax.axis("off")
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


## 7. Select best checkpoint and complete Phase 4 analysis

In [ ]:
if metric_table.empty:
    env_ckpt = os.environ.get("BEST_CKPT", "")
    if not env_ckpt:
        raise RuntimeError("No trained runs available. Set RUN_FULL_TRAINING=1 or BEST_CKPT=/path/to/best.pt.")
    best_ckpt = Path(env_ckpt)
    metric_table = pd.DataFrame([{"run": "external_best", "best_ckpt": str(best_ckpt), "imgsz": 640}])
else:
    best_row = metric_table.sort_values(["mAP50_95_mask", "mAP50_mask", "size_mb"], ascending=[False, False, True]).iloc[0]
    best_ckpt = Path(best_row["best_ckpt"])

print(f"Best checkpoint: {best_ckpt}")
CUSTOM_EVAL_MAX_IMAGES = int(os.environ.get("CUSTOM_EVAL_MAX_IMAGES", "0")) or None
custom_summary, custom_score_df = evaluate_custom_masks(best_ckpt, converted_df, imgsz=640, max_images=CUSTOM_EVAL_MAX_IMAGES)
speed_ms = benchmark_inference(best_ckpt, converted_df, imgsz=640, n_images=int(os.environ.get("BENCHMARK_IMAGES", "50")))
print(custom_summary)
print(f"Inference: {speed_ms:.2f} ms/image")

custom_score_df.to_csv(ARTIFACTS_DIR / "custom_mask_scores.csv", index=False)
summary_rows = metric_table.copy()
summary_rows["mIoU"] = custom_summary["mIoU"]
summary_rows["Dice"] = custom_summary["Dice"]
summary_rows["inference_ms_img"] = speed_ms
summary_rows.to_csv(ARTIFACTS_DIR / "phase34_yolo26_seg_summary.csv", index=False)
display(summary_rows)

plot_prediction_samples(best_ckpt, converted_df, n=10, title="YOLO26-seg predictions vs ground truth")

worst = custom_score_df.sort_values("mIoU").head(12)
display(worst[["sample_id", "mIoU", "Dice", "gt_classes", "pred_classes", "pair_error"]])
pair_errors = custom_score_df[custom_score_df["pair_error"]].sort_values("mIoU").head(10)
display(pair_errors[["sample_id", "mIoU", "Dice", "gt_classes", "pred_classes"]])


## 8. Export model and write lightweight artifacts

In [ ]:
best_model = YOLO(str(best_ckpt))
exported_onnx = best_model.export(format="onnx", imgsz=640, dynamic=True, simplify=True, opset=17)
exported_onnx = Path(exported_onnx)
print(f"Exported ONNX: {exported_onnx}")

shutil.copy2(best_ckpt, ARTIFACTS_DIR / "best.pt")
if exported_onnx.exists():
    shutil.copy2(exported_onnx, ARTIFACTS_DIR / "best.onnx")
(ARTIFACTS_DIR / "class_names.json").write_text(json.dumps(CLASS_NAMES, indent=2, ensure_ascii=False), encoding="utf-8")
shutil.copy2(DATA_YAML, ARTIFACTS_DIR / "data.yaml")

model_card = f"""# YOLO26-seg Plant Disease Segmentation

Owner: Nguyen Ho Anh Tuan

## Task
Instance segmentation for Vietnamese rice and coffee leaf disease images.

## Classes
{CLASS_NAMES}

## Best checkpoint
`{best_ckpt}`

## Metrics

- Custom mIoU: {custom_summary['mIoU']:.6f}
- Custom Dice: {custom_summary['Dice']:.6f}
- Inference: {speed_ms:.2f} ms/image

See `phase34_yolo26_seg_summary.csv` and `custom_mask_scores.csv` for full results.
"""
(ARTIFACTS_DIR / "README.md").write_text(model_card, encoding="utf-8")
print(f"Artifacts written to {ARTIFACTS_DIR}")
print(sorted(p.name for p in ARTIFACTS_DIR.iterdir()))


## 9. Optional HuggingFace Hub upload

Set `HF_REPO_ID`, `HF_TOKEN`, and `HF_UPLOAD=1` in Kaggle secrets/environment before running this cell. The default repo id is a placeholder and must be changed.

In [ ]:
HF_REPO_ID = os.environ.get("HF_REPO_ID", "<team-or-user>/ml-vietnam-plant-disease-yolo26-seg")
HF_UPLOAD = os.environ.get("HF_UPLOAD", "0") == "1"

if HF_UPLOAD:
    from huggingface_hub import HfApi, create_repo, upload_folder

    token = os.environ.get("HF_TOKEN")
    if not token:
        raise RuntimeError("Set HF_TOKEN before uploading to HuggingFace Hub.")
    if HF_REPO_ID.startswith("<team-or-user>"):
        raise RuntimeError("Set HF_REPO_ID to the real HuggingFace repo id before upload.")
    create_repo(repo_id=HF_REPO_ID, repo_type="model", private=False, exist_ok=True, token=token)
    upload_folder(
        repo_id=HF_REPO_ID,
        repo_type="model",
        folder_path=str(ARTIFACTS_DIR),
        commit_message="Add YOLO26-seg phase 3+4 artifacts",
        token=token,
    )
    print(f"Uploaded artifacts to https://huggingface.co/{HF_REPO_ID}")
else:
    print("HF_UPLOAD=0, skipping upload.")
    print(f"When ready, set HF_REPO_ID={HF_REPO_ID!r}, HF_TOKEN, and HF_UPLOAD=1.")
